# Program 11 – Comparative Study of Activation and Regularization Techniques

**Objective:** Train and compare four versions of the same softmax-based MLP on MNIST to study how activation, normalization and regularization affect performance.

**Dataset:** MNIST

| Model | Change | What to look for |
|---|---|---|
| 1 | Sigmoid activation (2 hidden layers) | slow training, vanishing gradient |
| 2 | ReLU activation | faster convergence, higher accuracy than Model 1 |
| 3 | ReLU + `BatchNormalization()` after each Dense | better stability and accuracy |
| 4 | ReLU + `Dropout(0.3–0.5)` | smaller train/validation gap (less overfitting) |

All models use a **softmax output layer** for 10 classes.

## Theory (for viva)

**Sigmoid** $\sigma(x) = \frac{1}{1+e^{-x}}$, derivative $\sigma'(x) = \sigma(x)(1-\sigma(x)) \le 0.25$.
During backprop each layer multiplies the gradient by ≤ 0.25 → gradients in early layers become tiny → **vanishing gradient** → slow learning. Outputs are also not zero-centred.

**ReLU** $f(x) = \max(0, x)$, derivative = 1 for $x>0$ → gradients pass through unchanged → faster training. Risk: "dead" neurons for $x<0$.

**Batch Normalization** normalizes each layer's outputs over the mini-batch (mean 0, variance 1) and then applies learnable scale $\gamma$ and shift $\beta$. Reduces internal covariate shift, allows higher learning rates, adds slight regularization.

**Dropout(p)** randomly zeroes a fraction $p$ of neurons in each training step → prevents co-adaptation → reduces overfitting. Turned off at test time.

**Fair comparison:** same layer sizes, optimizer, learning rate, epochs, batch size and seed; only one technique changes at a time.

## Step 1 – Import libraries

In [ ]:
import keras  # Keras 3 API (runs on TensorFlow backend)
from keras import layers  # Layer classes (Dense, Dropout, Embedding, ...)
import matplotlib.pyplot as plt  # Plotting graphs
import numpy as np  # Numerical arrays
import tensorflow as tf  # TensorFlow (tensors, GradientTape, seeds)

## Step 2 – Load and normalize MNIST

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()  # Load MNIST
x_train = x_train.reshape(-1, 784).astype("float32") / 255.0              # Flatten + scale to [0, 1]
x_test = x_test.reshape(-1, 784).astype("float32") / 255.0                # Flatten + scale to [0, 1]
y_train_cat = keras.utils.to_categorical(y_train, 10)                     # One-hot labels for categorical CE
y_test_cat = keras.utils.to_categorical(y_test, 10)                       # One-hot test labels
print(x_train.shape, y_train_cat.shape)                                   # (60000, 784) (60000, 10)

## Step 3 – Define the four models

In [ ]:
H1, H2 = 512, 256                                                   # Sizes of the two hidden layers (same for all models)


def model_1_sigmoid():                                              # Model 1 – Sigmoid activation
    return keras.Sequential([                                       # Sequential stack
        keras.Input(shape=(784,)),                                  # Input
        layers.Dense(H1, activation="sigmoid"),                     # Hidden 1 with sigmoid
        layers.Dense(H2, activation="sigmoid"),                     # Hidden 2 with sigmoid
        layers.Dense(10, activation="softmax"),                     # Softmax output
    ], name="M1_Sigmoid")                                           # Model name


def model_2_relu():                                                 # Model 2 – ReLU activation
    return keras.Sequential([                                       # Sequential stack
        keras.Input(shape=(784,)),                                  # Input
        layers.Dense(H1, activation="relu"),                        # Hidden 1 with ReLU
        layers.Dense(H2, activation="relu"),                        # Hidden 2 with ReLU
        layers.Dense(10, activation="softmax"),                     # Softmax output
    ], name="M2_ReLU")                                              # Model name


def model_3_batchnorm():                                            # Model 3 – ReLU + Batch Normalization
    return keras.Sequential([                                       # Sequential stack
        keras.Input(shape=(784,)),                                  # Input
        layers.Dense(H1, activation="relu"),                        # Hidden 1 with ReLU
        layers.BatchNormalization(),                                # BN after Dense layer 1
        layers.Dense(H2, activation="relu"),                        # Hidden 2 with ReLU
        layers.BatchNormalization(),                                # BN after Dense layer 2
        layers.Dense(10, activation="softmax"),                     # Softmax output
    ], name="M3_BatchNorm")                                         # Model name


def model_4_dropout():                                              # Model 4 – ReLU + Dropout
    return keras.Sequential([                                       # Sequential stack
        keras.Input(shape=(784,)),                                  # Input
        layers.Dense(H1, activation="relu"),                        # Hidden 1 with ReLU
        layers.Dropout(0.4),                                        # Drop 40% of neurons during training
        layers.Dense(H2, activation="relu"),                        # Hidden 2 with ReLU
        layers.Dropout(0.4),                                        # Drop 40% of neurons during training
        layers.Dense(10, activation="softmax"),                     # Softmax output
    ], name="M4_Dropout")                                           # Model name


model_builders = {                                                  # Map display name -> builder function
    "Model 1 - Sigmoid": model_1_sigmoid,                           # Sigmoid
    "Model 2 - ReLU": model_2_relu,                                 # ReLU
    "Model 3 - BatchNorm": model_3_batchnorm,                       # BatchNorm
    "Model 4 - Dropout": model_4_dropout,                           # Dropout
}

## Step 4 – Train all four models with identical settings

In [ ]:
EPOCHS = 20                                                         # Epochs for each model
BATCH_SIZE = 128                                                    # Batch size

histories, results, trained = {}, {}, {}                            # Store history, test results and models

for name, builder in model_builders.items():                        # Loop over models
    print(f"\n========== {name} ==========")                        # Header
    tf.random.set_seed(42)                                          # Same seed for every model
    model = builder()                                               # Build model
    model.compile(                                                  # Compile with SAME settings for all
        optimizer=keras.optimizers.SGD(learning_rate=0.01,          # SGD makes activation differences visible
                                       momentum=0.9),               # Momentum speeds up SGD
        loss="categorical_crossentropy",                            # One-hot labels
        metrics=["accuracy"],                                       # Accuracy
    )
    h = model.fit(x_train, y_train_cat,                             # Train
                  epochs=EPOCHS, batch_size=BATCH_SIZE,             # Epochs & batch size
                  validation_split=0.2, verbose=2)                  # 20% validation
    histories[name] = h.history                                     # Save history
    results[name] = model.evaluate(x_test, y_test_cat, verbose=0)   # [test_loss, test_acc]
    trained[name] = model                                           # Keep trained model

## Step 5 – Compare validation accuracy and loss of all models

In [ ]:
ep = range(1, EPOCHS + 1)                                           # X-axis
fig, axes = plt.subplots(1, 2, figsize=(16, 5))                     # Two plots
for name, h in histories.items():                                   # One line per model
    axes[0].plot(ep, h["val_accuracy"], "o-", label=name)           # Validation accuracy
    axes[1].plot(ep, h["val_loss"], "o-", label=name)               # Validation loss
axes[0].set_title("Validation Accuracy")                            # Title
axes[1].set_title("Validation Loss")                                # Title
for ax in axes:                                                     # Decorate both
    ax.set_xlabel("Epoch")                                          # X label
    ax.legend()                                                     # Legend
    ax.grid(True)                                                   # Grid
plt.tight_layout()                                                  # Tidy
plt.show()                                                          # Show

## Step 6 – Training vs validation curves for each model (overfitting check)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 9))                     # 2 rows (acc, loss) x 4 models
for col, (name, h) in enumerate(histories.items()):                 # One column per model
    axes[0, col].plot(ep, h["accuracy"], label="Train")             # Train accuracy
    axes[0, col].plot(ep, h["val_accuracy"], label="Validation")    # Validation accuracy
    axes[0, col].set_title(f"{name}\nAccuracy")                     # Title
    axes[0, col].legend()                                           # Legend
    axes[0, col].grid(True)                                         # Grid
    axes[1, col].plot(ep, h["loss"], label="Train")                 # Train loss
    axes[1, col].plot(ep, h["val_loss"], label="Validation")        # Validation loss
    axes[1, col].set_title("Loss")                                  # Title
    axes[1, col].set_xlabel("Epoch")                                # X label
    axes[1, col].legend()                                           # Legend
    axes[1, col].grid(True)                                         # Grid
plt.tight_layout()                                                  # Tidy
plt.show()                                                          # Show

## Step 7 – Check for vanishing gradient (Model 1 vs Model 2)

Compute the gradient norm of each Dense layer's weights on one batch. Much smaller first-layer gradients for sigmoid = vanishing gradient.

In [ ]:
x_batch, y_batch = x_train[:256], y_train_cat[:256]                 # One batch of data
loss_fn = keras.losses.CategoricalCrossentropy()                    # Same loss as training


def layer_grad_norms(model):                                        # Returns gradient norm of each Dense kernel
    with tf.GradientTape() as tape:                                 # Record forward pass
        loss = loss_fn(y_batch, model(x_batch, training=False))     # Compute loss
    dense_kernels = [ly.kernel for ly in model.layers               # Kernel (weight matrix) of each Dense layer
                     if isinstance(ly, layers.Dense)]               # Only Dense layers
    grads = tape.gradient(loss, dense_kernels)                      # d(loss)/d(kernel)
    return [float(tf.norm(g)) for g in grads]                       # L2 norm per layer


fig, axes = plt.subplots(1, 2, figsize=(14, 4))                     # Two plots
for ax, when in zip(axes, ["untrained (initial)", "trained"]):      # Compare before and after training
    for name in ["Model 1 - Sigmoid", "Model 2 - ReLU"]:            # Only sigmoid vs ReLU
        if when == "trained":                                       # Trained case
            m = trained[name]                                       # Use trained model
        else:                                                       # Untrained case
            tf.random.set_seed(42)                                  # Same initial weights as training run
            m = model_builders[name]()                              # Fresh untrained model
        norms = layer_grad_norms(m)                                 # Gradient norm per layer
        ax.plot(["Hidden 1", "Hidden 2", "Output"], norms, "o-", label=name)  # Plot per layer
        print(f"{when:<20} {name:<18} grad norms per layer: {np.round(norms, 5)}")  # Print values
    ax.set_yscale("log")                                            # Log scale
    ax.set_title(f"Gradient norm per layer – {when}")               # Title
    ax.set_ylabel("||dL/dW||")                                      # Y label
    ax.legend()                                                     # Legend
    ax.grid(True)                                                   # Grid
plt.tight_layout()                                                  # Tidy
plt.show()                                                          # Show

## Step 8 – Summary table

In [ ]:
print(f"{'Model':<22} {'Train acc':>9} {'Val acc':>8} {'Gap':>7} {'Test acc':>9} {'Epoch-1 val acc':>16}")  # Header
print("-" * 78)                                                     # Separator
for name, h in histories.items():                                   # One row per model
    train_acc = h["accuracy"][-1]                                   # Final training accuracy
    val_acc = h["val_accuracy"][-1]                                 # Final validation accuracy
    gap = train_acc - val_acc                                       # Overfitting gap (train - val)
    test_acc = results[name][1]                                     # Test accuracy
    first = h["val_accuracy"][0]                                    # Validation accuracy after epoch 1 (convergence speed)
    print(f"{name:<22} {train_acc:>9.4f} {val_acc:>8.4f} {gap:>7.4f} {test_acc:>9.4f} {first:>16.4f}")  # Print row

In [ ]:
names = list(results.keys())                                        # Model names
accs = [results[n][1] for n in names]                               # Test accuracies
plt.figure(figsize=(10, 5))                                         # Figure
bars = plt.bar(names, accs, color=["tab:gray", "tab:blue", "tab:green", "tab:orange"])  # Bars
for bar, a in zip(bars, accs):                                      # Value labels
    plt.text(bar.get_x() + bar.get_width() / 2, a + 0.001, f"{a:.4f}", ha="center")  # Text above bar
plt.ylim(min(accs) - 0.02, 1.0)                                     # Zoom y-axis
plt.ylabel("Test accuracy")                                         # Y label
plt.title("Test accuracy of the four models")                       # Title
plt.show()                                                          # Show

## Expected observations
- **Model 1 (Sigmoid):** slowest start (lowest epoch-1 accuracy); first-layer gradients are smaller than ReLU's → mild vanishing gradient (only 2 hidden layers, so it still learns, just slower). Test ≈ 97%.
- **Model 2 (ReLU):** converges much faster, ≈ 98%; training accuracy approaches 100% → some overfitting (train–val gap grows).
- **Model 3 (BatchNorm):** smooth and fast; often highest accuracy in few epochs; validation loss may be a bit noisy.
- **Model 4 (Dropout):** training accuracy lower than Model 2, but smallest train–val gap → best generalization.

## Viva questions
1. Why does sigmoid cause vanishing gradients? → Its derivative is at most 0.25, and near 0 for large |x| (saturation).
2. Why is ReLU faster? → Derivative is 1 for positive inputs; cheap to compute; no saturation for x > 0.
3. What is internal covariate shift? → Change in distribution of layer inputs during training; BN reduces it.
4. Where is BN placed? → After Dense (before or after activation — both are used; this program places it after the activated Dense output).
5. Why is validation accuracy sometimes higher than training accuracy with dropout? → Dropout is active only during training; validation uses the full network.
6. Typical dropout rate? → 0.2–0.5 for dense layers.